[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [asyncpg and psycopg3, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/asyncpg-and-psycopg3-deep-dive.html)

# A Server of Your Own &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell is the notebook's Setup: a server, the `guide` database and five thousand events. Run
it first. Tasks 5 and 6 make a table of their own, so run those two in order.


In [1]:
import getpass
import os
import sqlite3
import subprocess
import sys
import tempfile
import time
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

try:
    if version("psycopg") < "3.3" or version("asyncpg") < "0.31":
        raise PackageNotFoundError
except PackageNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "psycopg[binary,pool]==3.3.6", "psycopg-pool==3.3.2", "asyncpg==0.31.0"],
                   check=True)

import asyncpg
import psycopg

def shell(command):
    """Run a shell command and hand back what it printed, without letting it stop the notebook."""
    done = subprocess.run(command, shell=True, capture_output=True, text=True)
    return done.returncode, (done.stdout + done.stderr).strip()


def answering(database="postgres"):
    """Whether a server is there, asked the only way that needs no client binaries."""
    try:
        with psycopg.connect(f"dbname={database}", connect_timeout=2):
            return True
    except psycopg.OperationalError:
        return False


def start_server(wait=60):
    """Install and start PostgreSQL if nothing is answering. Returns what it had to do."""
    if answering():
        return "already running"
    if sys.platform != "linux":
        raise RuntimeError("No PostgreSQL is answering. Start your own server and run this again: "
                           "this cell only installs one on Linux, which is what Colab runs.")

    sudo = "" if os.geteuid() == 0 else "sudo "
    shell(f"{sudo}apt-get -qq update")
    shell(f"{sudo}apt-get -qq -y install postgresql postgresql-contrib")
    shell(f"{sudo}service postgresql start")                        # Colab has no systemd

    for attempt in range(1, wait + 1):                              # start returns before it listens
        if shell("pg_isready -q")[0] == 0:
            break
        print(f"  waiting for the cluster ({attempt})")              # a silent minute looks hung
        time.sleep(1)
    else:
        raise RuntimeError(f"PostgreSQL did not accept connections within {wait} seconds.")

    me = getpass.getuser()                                          # peer authentication wants a role
    asking = f"""sudo -u postgres psql -tAc "SELECT 1 FROM pg_roles WHERE rolname='{me}'" """
    if shell(asking)[1] != "1":                                     # named for the operating system user
        shell(f"sudo -u postgres createuser -s {me}")
    return "installed and started"

def build(rows=5000):
    """Make the guide database and its events table, and fill it once."""
    with psycopg.connect("dbname=postgres", autocommit=True) as conn:
        if not conn.execute("SELECT 1 FROM pg_database WHERE datname = 'guide'").fetchone():
            conn.execute("CREATE DATABASE guide")                   # cannot run in a transaction

    with psycopg.connect("dbname=guide", autocommit=True) as conn:
        for (leftover,) in conn.execute(                            # whatever an earlier run made
                "SELECT tablename FROM pg_tables "
                "WHERE schemaname = 'public' AND tablename <> 'events'").fetchall():
            conn.execute(f'DROP TABLE IF EXISTS "{leftover}" CASCADE')

        conn.execute("""CREATE TABLE IF NOT EXISTS events (
                            id bigserial PRIMARY KEY,
                            ts timestamptz NOT NULL DEFAULT now(),
                            kind text NOT NULL,
                            payload jsonb NOT NULL)""")
        if conn.execute("SELECT count(*) FROM events").fetchone()[0] == 0:
            conn.execute("""INSERT INTO events (kind, payload)
                            SELECT (ARRAY['click', 'view', 'purchase'])[1 + n %% 3],
                                   jsonb_build_object('n', n, 'size', 1 + n %% 7)
                            FROM generate_series(1, %s) AS n""", (rows,))
        return conn.execute("SELECT count(*) FROM events").fetchone()[0]

def report():
    """One line naming what this notebook is running against."""
    rows = build()                                                  # makes the database if it is new
    with psycopg.connect("dbname=guide") as conn:
        major = int(conn.execute("SHOW server_version_num").fetchone()[0]) // 10000
    return (f"PostgreSQL {major} | psycopg {version('psycopg')} | asyncpg {version('asyncpg')} "
            f"| events: {rows} rows")

def fatal(error):
    """What the server said, without the socket path, which is different on every machine."""
    line = str(error).strip().splitlines()[0]
    return line.rsplit("failed: ", 1)[-1]


print("server:", start_server())
print(report())


server: already running
PostgreSQL 16 | psycopg 3.3.6 | asyncpg 0.31.0 | events: 5000 rows


**1.** Three questions, one query.


In [2]:
with psycopg.connect("dbname=guide") as conn:
    number, database, who = conn.execute(
        "SELECT current_setting('server_version_num'), current_database(), current_user").fetchone()

print("server version:", int(number) // 10000)
print("database:      ", database)
print("role:          ", "the operating system user" if who == getpass.getuser() else who)


server version: 16
database:       guide
role:           the operating system user


All three come from the session rather than from anything you passed in, which is the point: they
say what you actually got, not what you asked for.


**2.** Two lists, and the word that covers both.


In [3]:
with psycopg.connect("dbname=guide") as conn:
    databases = [row[0] for row in conn.execute(
        "SELECT datname FROM pg_database WHERE NOT datistemplate ORDER BY datname")]
    tables = [row[0] for row in conn.execute(
        "SELECT tablename FROM pg_tables WHERE schemaname = 'public' ORDER BY tablename")]

print("databases in the cluster:", databases)
print("tables in guide:         ", tables)


databases in the cluster: ['guide', 'postgres']
tables in guide:          ['events']


The first list is databases inside one cluster, so "the database" there means one of several names.
The second is tables inside one of those, so "the database" there means the thing holding them. The
connection was open to `guide` the whole time, which is why the second list did not have to say so.


**3.** The same count, from both drivers.


In [4]:
with psycopg.connect("dbname=guide") as conn:
    synchronous = conn.execute("SELECT count(*) FROM events").fetchone()[0]

connection = await asyncpg.connect(database="guide")
asynchronous = await connection.fetchval("SELECT count(*) FROM events")
await connection.close()

print("psycopg:", synchronous, "| asyncpg:", asynchronous, "| agree:", synchronous == asynchronous)


psycopg: 5000 | asyncpg: 5000 | agree: True


Two drivers, two protocols' worth of code, one server and one answer. `fetchval` is asyncpg's way of
saying "one row, one column", which psycopg spells `.fetchone()[0]`.


**4.** A database that is not there.


In [5]:
try:
    psycopg.connect("dbname=nowhere", connect_timeout=2)
except psycopg.OperationalError as error:
    print("the whole message has a socket path in it, which is different on every machine")
    print("what the server actually said:", fatal(error))


the whole message has a socket path in it, which is different on every machine
what the server actually said: FATAL:  database "nowhere" does not exist


`fatal` keeps the part after the last `failed: `, which is the server's own sentence. The rest of the
line is libpq saying where it looked.


**5.** Two writers, two rows.


In [6]:
first = psycopg.connect("dbname=guide")
second = psycopg.connect("dbname=guide")

first.execute("CREATE TABLE IF NOT EXISTS notes (id int PRIMARY KEY, body text)")
first.commit()
first.execute("TRUNCATE notes")
first.commit()

first.execute("INSERT INTO notes VALUES (1, 'from the first')")     # neither has committed
second.execute("INSERT INTO notes VALUES (2, 'from the second')")
print("both wrote, neither waited")

first.commit()
second.commit()
print("rows:", first.execute("SELECT id, body FROM notes ORDER BY id").fetchall())


both wrote, neither waited
rows: [(1, 'from the first'), (2, 'from the second')]


Different rows, different locks, no waiting. Had they both written row 1, the second would have
waited for the first to commit or roll back, which is the next task.


**6.** Two writers, one row.


In [7]:
first.execute("UPDATE notes SET body = 'first again' WHERE id = 1")

second.execute("SET lock_timeout = '500ms'")                        # so the cell cannot hang
try:
    second.execute("UPDATE notes SET body = 'second too' WHERE id = 1")
except psycopg.errors.LockNotAvailable as error:
    print("the second writer waited, then gave up:", fatal(error))

second.rollback()
first.rollback()
print("rows, unchanged:", first.execute("SELECT id, body FROM notes ORDER BY id").fetchall())
first.close()
second.close()


the second writer waited, then gave up: canceling statement due to lock timeout
rows, unchanged: [(1, 'from the first'), (2, 'from the second')]


The row is what was locked, and the timeout is what turned waiting into an error. Without it the
second `UPDATE` would have sat there until the first connection committed or rolled back, which is
the ordinary behavior and is usually what you want.


---

&#8592; **Back to:** [A Server of Your Own](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/asyncpg-and-psycopg3-deep-dive/01-a-server-of-your-own.ipynb)  &nbsp;&middot;&nbsp;  [asyncpg and psycopg3, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/asyncpg-and-psycopg3-deep-dive.html)
